# Agent Intro: Tool Calling with Groq or OpenRouter

This notebook demonstrates how an LLM can call a Python function (a tool) to answer a question.

**Important**
- Keep API keys in a `.env` file. Never paste them into notebook cells or commit `.env` to Git.
- Use a real Groq key for `GROQ_API_KEY` and a real OpenRouter key for `OPENROUTER_API_KEY`.
- Set `LLM_MODEL` to a model ID available from the provider selected by `LLM_PROVIDER`.
- The weather data below is sample data stored in this notebook; it is not live weather.

## 1. Install dependencies

Run this cell once in the notebook environment if the packages are not already installed.

In [1]:
%pip install -U openai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configure `.env`

Create a file named `.env` in the same folder as this notebook:

```env
GROQ_API_KEY=your_real_groq_key
OPENROUTER_API_KEY=your_real_openrouter_key
LLM_PROVIDER=OpenRouter
LLM_MODEL=your_model_id_from_openrouter
```

For Groq, change `LLM_PROVIDER=Groq` and set `LLM_MODEL` to a model ID available in your Groq account.

Do not put the same key in both variables unless it is genuinely a valid key for both providers.

In [12]:
from dotenv import load_dotenv
from pathlib import Path
import os
from openai import OpenAI
import json

load_dotenv(Path.cwd() / ".env", override=True)

model = os.getenv("LLM_MODEL", "").strip()
api_key = os.getenv("GROQ_API_KEY", "").strip()

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

print("Model:", model)

Model: openai/gpt-oss-20b


In [13]:
response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "user", "content": "Say hello in one sentence."}
    ],
    max_tokens=100
)

print(response.choices[0].message.content)

Hello! How can I help you today?


## 3. Define a local weather tool

This function looks up sample data. In a real project, the function could call a weather API.

In [14]:
WEATHER_DB = {
    "london": {"temperature_c": 22, "condition": "cloudy"},
    "paris": {"temperature_c": 24, "condition": "sunny"},
    "tokyo": {"temperature_c": 30, "condition": "sunny"},
    "mumbai": {"temperature_c": 32, "condition": "sunny"},
}

def lookup_weather(location: str) -> dict:
    """Return sample weather data for a supported city."""
    city = location.strip().lower()
    weather = WEATHER_DB.get(city)

    if weather is None:
        return {
            "location": location,
            "error": "City not found in sample weather data.",
            "available_cities": sorted(WEATHER_DB.keys()),
        }

    return {"location": location.title(), **weather}

# Try the Python function directly before connecting it to the model.
lookup_weather("Tokyo")

{'location': 'Tokyo', 'temperature_c': 30, 'condition': 'sunny'}

## 4. Describe the tool to the model

The tool schema tells the model the function name, purpose, and required argument.

In [15]:
weather_tool = {
    "type": "function",
    "function": {
        "name": "lookup_weather",
        "description": "Look up sample weather data for a supported city.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city name, such as Tokyo or London."
                }
            },
            "required": ["location"],
            "additionalProperties": False
        }
    }
}

TOOLS = {"lookup_weather": lookup_weather}

## 5. Build the tool-calling agent

The loop works as follows:
1. Send the user question and tool schema to the model.
2. If the model requests a tool, run the matching Python function.
3. Send the tool result back to the model.
4. Repeat until the model returns a normal answer.

The loop has a maximum number of rounds to avoid running forever.

In [16]:
def ask_agent(prompt: str, max_rounds: int = 5) -> str:
    messages = [{"role": "user", "content": prompt}]

    for _ in range(max_rounds):
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=[weather_tool],
            tool_choice="auto",
            temperature=0,
            max_tokens=500,
        )

        message = response.choices[0].message

        # No tool call: the model has produced its final response.
        if not message.tool_calls:
            return message.content or "(The model returned an empty response.)"

        # Preserve the assistant's tool-call message in the conversation.
        messages.append(message.model_dump(exclude_none=True))

        # Execute each requested tool and append its result.
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name

            if function_name not in TOOLS:
                tool_result = {"error": f"Unknown tool: {function_name}"}
            else:
                try:
                    arguments = json.loads(tool_call.function.arguments or "{}")
                    tool_result = TOOLS[function_name](**arguments)
                except (json.JSONDecodeError, TypeError) as exc:
                    tool_result = {"error": f"Invalid tool arguments: {exc}"}
                except Exception as exc:
                    tool_result = {"error": f"Tool execution failed: {exc}"}

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(tool_result),
            })

    return "Stopped because the maximum tool-calling rounds were reached." 

## 6. Ask the agent a question

In [17]:
print(ask_agent("What is the weather in Tokyo?"))

Here’s the current weather in Tokyo:

- **Temperature:** 30 °C  
- **Condition:** Sunny  

Let me know if you’d like more details (e.g., humidity, wind, forecast).


## 7. Try a comparison

In [18]:
print(ask_agent("Compare the sample weather in London and Paris."))

**Sample Weather Comparison**

| City   | Temperature (°C) | Condition |
|--------|------------------|-----------|
| London | 22 °C            | Cloudy    |
| Paris  | 24 °C            | Sunny     |

**Key Takeaways**

- **Temperature**: Paris is slightly warmer than London by 2 °C.
- **Weather Condition**: London is overcast, while Paris enjoys clear, sunny skies.


## 8. Try a city that is not in the sample data

In [19]:
print(ask_agent("What is the weather in Delhi?"))

I’m sorry, but I don’t have weather data for Delhi in the sample dataset. The available cities in the sample are:

- London  
- Mumbai  
- Paris  
- Tokyo  

If you’d like the weather for one of those cities—or if you have another city in mind that’s in the dataset—just let me know!


## 9. Troubleshooting

- **401 / invalid API key:** Create a fresh key in the dashboard for the selected provider, update the matching `.env` variable, and rerun the setup cell. Do not share the key.
- **Model not found:** Set `LLM_MODEL` to an exact model ID available for the selected provider.
- **Missing key:** Check that `.env` is in the notebook's current working directory. Run `Path.cwd()` to see that directory.
- **Tool calls not supported:** Select a model that supports tool/function calling on your provider.
- **OpenRouter:** The key must be an OpenRouter key, and the base URL must be `https://openrouter.ai/api/v1`.
- **Groq:** The key must be a Groq key, and the base URL must be `https://api.groq.com/openai/v1`.

To check the notebook's working directory:

```python
print(Path.cwd())
print(".env exists:", (Path.cwd() / ".env").exists())
```

Never print the complete API key.

## What you learned

- How to load API configuration from `.env`.
- How to switch between Groq and OpenRouter using `LLM_PROVIDER`.
- How to define a Python function as a tool.
- How to describe a tool with a JSON schema.
- How to execute model-requested tools and send results back to the model.

**Reminder:** This is a small educational agent. Its weather values are hard-coded examples, not current conditions.